# Complexidade assintótica e MergeSort

Este notebook reúne duas partes da atividade:

1. análise de afirmações sobre complexidade assintótica;
2. simulação das primeiras etapas do MergeSort sobre o vetor:

```python
A = [12, 7, 3, 15, 9, 1, 8, 4]
```

In [1]:
%load_ext cython

## Parte 1 — Afirmações sobre complexidade

### Afirmação I

> Se um algoritmo A possui complexidade `O(n²)` e um algoritmo B possui complexidade `O(n³)`, então A sempre será mais rápido que B.

**Falsa.** A notação assintótica descreve crescimento para valores suficientemente grandes de `n`, mas constantes e detalhes de implementação podem fazer um algoritmo `O(n³)` ser mais rápido para entradas pequenas.

Contraexemplo:

\[
A(n)=1000n^2
\]

\[
B(n)=n^3
\]

Para `n=3`:

\[
A(3)=9000, \quad B(3)=27
\]

Logo, A não é sempre mais rápido.

### Afirmação II

> Se `T(n) = 4n² + 3n + 10`, então `T(n) = Θ(n²)`.

**Verdadeira.** O termo dominante é `n²`.

### Afirmação III

> Se `T(n) = n log n`, então também é correto afirmar que `T(n) = O(n²)`.

**Verdadeira.** Como `n log n` cresce mais lentamente que `n²`, `n²` é um limite superior assintótico válido.

### Afirmação IV

> Se dois algoritmos têm a mesma complexidade assintótica, então necessariamente apresentam o mesmo tempo de execução na prática.

**Falsa.** Dois algoritmos podem ser `Θ(n²)` e ter constantes muito diferentes.

Exemplo:

\[
T_1(n)=2n^2
\]

\[
T_2(n)=10n^2
\]

Ambos são `Θ(n²)`, mas `T₂` executa mais operações.

In [2]:
def A(n):
    return 1000 * n**2

def B(n):
    return n**3

for n in [3, 10, 1000]:
    print(f"n={n:4d} | A(n)={A(n):12d} | B(n)={B(n):12d}")

n=   3 | A(n)=        9000 | B(n)=          27
n=  10 | A(n)=      100000 | B(n)=        1000
n=1000 | A(n)=  1000000000 | B(n)=  1000000000


## Parte 2 — MergeSort: duas primeiras etapas

Vetor inicial:

```python
[12, 7, 3, 15, 9, 1, 8, 4]
```

Primeira divisão:

```text
[12, 7, 3, 15]    [9, 1, 8, 4]
```

Segunda divisão:

```text
[12, 7] [3, 15]   [9, 1] [8, 4]
```

Depois disso, o algoritmo continua dividindo até subvetores de tamanho 1 e começa a intercalação ordenada.

## Análise assintótica do MergeSort

O MergeSort divide o vetor em duas metades e depois intercala os resultados em tempo linear.

A recorrência é:

\[
T(n)=2T(n/2)+\Theta(n)
\]

Pelo Teorema Mestre:

\[
T(n)=\Theta(n\log n)
\]

O espaço auxiliar é **Θ(n)** por causa dos vetores temporários usados na intercalação.

In [3]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

cdef void _merge(list A, Py_ssize_t esquerda, Py_ssize_t meio, Py_ssize_t direita):
    cdef list L = A[esquerda:meio + 1]
    cdef list R = A[meio + 1:direita + 1]
    cdef Py_ssize_t i = 0, j = 0, k = esquerda
    cdef Py_ssize_t nL = len(L), nR = len(R)

    while i < nL and j < nR:
        if L[i] <= R[j]:
            A[k] = L[i]
            i += 1
        else:
            A[k] = R[j]
            j += 1
        k += 1

    while i < nL:
        A[k] = L[i]
        i += 1
        k += 1

    while j < nR:
        A[k] = R[j]
        j += 1
        k += 1

cdef void _mergesort(list A, Py_ssize_t esquerda, Py_ssize_t direita):
    cdef Py_ssize_t meio
    if esquerda >= direita:
        return
    meio = (esquerda + direita) // 2
    _mergesort(A, esquerda, meio)
    _mergesort(A, meio + 1, direita)
    _merge(A, esquerda, meio, direita)

def mergesort(list entrada):
    cdef list A = list(entrada)
    if len(A) > 1:
        _mergesort(A, 0, len(A) - 1)
    return A

Content of stdout:
_cython_magic_05c28e118f382cdf2ef6dc63fe6d294f3da53f31601faeabf789966ed076c158.c
   Criando biblioteca C:\Users\Leandro Tosta\.ipython\cython\Users\Leandro Tosta\.ipython\cython\_cython_magic_05c28e118f382cdf2ef6dc63fe6d294f3da53f31601faeabf789966ed076c158.cp312-win_amd64.lib e objeto C:\Users\Leandro Tosta\.ipython\cython\Users\Leandro Tosta\.ipython\cython\_cython_magic_05c28e118f382cdf2ef6dc63fe6d294f3da53f31601faeabf789966ed076c158.cp312-win_amd64.exp
Gerando c¢digo
Finalizada a geraÆo de c¢digo

In [4]:
A = [12, 7, 3, 15, 9, 1, 8, 4]
print("Entrada: ", A)
print("Ordenado:", mergesort(A))

Entrada:  [12, 7, 3, 15, 9, 1, 8, 4]
Ordenado: [1, 3, 4, 7, 8, 9, 12, 15]


## Simulação visual das divisões

A função abaixo não é Cython porque serve apenas para exibir a árvore de divisão em sala. A implementação eficiente do algoritmo está na célula Cython anterior.

In [5]:
def mostrar_divisoes(A, nivel=0, max_nivel=2):
    print("  " * nivel + str(A))
    if nivel >= max_nivel or len(A) <= 1:
        return
    meio = len(A) // 2
    mostrar_divisoes(A[:meio], nivel + 1, max_nivel)
    mostrar_divisoes(A[meio:], nivel + 1, max_nivel)

mostrar_divisoes([12, 7, 3, 15, 9, 1, 8, 4])

[12, 7, 3, 15, 9, 1, 8, 4]
  [12, 7, 3, 15]
    [12, 7]
    [3, 15]
  [9, 1, 8, 4]
    [9, 1]
    [8, 4]
